In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

sns.set_style('whitegrid')

In [ ]:
#Figure 1 draws the domain heights the database uses: the 2026 measurement, one height per Pfam
#family (measurements/domains_2026/domain_heights.csv; height_basis says whether it is the neighbour
#pitch or a box dimension along a called axis, max_dim_len how many instances were measured), with
#the clan name from the table the height notebook takes each family's height from.  The 2020
#measurement it replaced is measurements/domains_2020.csv.
heights_2026 = pd.read_csv(resolve('measurements/domains_2026/domain_heights.csv'))
clan_names = pd.read_csv(resolve('inputs/domains/domain_clan_heights.csv'))[['Pfam', 'clan_name']].drop_duplicates()
domains = heights_2026.merge(clan_names, on = 'Pfam', how = 'left')
domains = domains[domains.height.notnull() & domains.clan_name.notnull()]
domains.height_basis.value_counts()


In [ ]:
sns.set(rc={'figure.figsize':(15,7)})
sns.set_style('whitegrid')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

clans = domains[domains.max_dim_len > 3].groupby('clan_name').count()[domains.groupby('clan_name').count()['height'] > 1].index

grouped = domains[domains.clan_name.isin(clans)&(domains.max_dim_len > 3)].groupby("clan_name")
users_sorted_average = pd.DataFrame({col:vals['height'] for col,vals in grouped}).median().sort_values(ascending=True) 

fig = sns.boxplot(x = 'clan_name', y = 'height', data = domains[domains.clan_name.isin(clans) & (domains.max_dim_len > 3)], 
                  order=users_sorted_average.index, fliersize = 3)

fig.set_xticklabels(fig.get_xticklabels(), rotation=45, horizontalalignment='right', fontweight='light')

plt.savefig(out_path('figures/F1_domain_sizes.pdf'))

In [ ]:
#Pfam 38 merged the Ig clan (CL0011) into E-set (CL0159), so the immunoglobulin families are
#drawn under E-set here where the 2020 figure had them as their own clan.
most_common = domains.clan_name.isin(['EGF', 'E-set', 'Beta_propeller', 'CUB', 'C_Lectin'])


In [ ]:
sns.set(rc={'figure.figsize':(5,5)})
sns.set_style('whitegrid')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sns.set_palette('husl', 4)

fig = sns.boxplot(x = 'clan_name', y = 'height', data = domains[most_common], fliersize = 3, order = ['EGF', 'E-set', 'Beta_propeller', 'CUB', 'C_Lectin'])

fig.set_xticklabels(fig.get_xticklabels(), rotation=45, horizontalalignment='right', fontweight='light')

plt.savefig(out_path('figures/F1_domain_sizes_most_common.pdf'))